# 📝 데이터베이스 기초 과제 LV1 — 학사 데이터로 익히는 SQL

교안에서는 **쇼핑몰**로 배웠습니다. 여기서는 **학사 데이터**(학과·학생·과목·수강)로 같은 개념을 적용해 봅니다. 데이터가 달라도 문법은 그대로라는 것을 손으로 확인하는 것이 목표입니다.

**풀이 방법**

1. 아래 준비 셀과 리셋 셀을 먼저 실행하세요.
2. 문제마다 **답안 셀**에 코드를 쓰고 실행한 뒤, 바로 아래 **자가채점 셀**을 실행하세요.
3. `✅ 통과!` 가 나오면 다음 문제로 갑니다. 막히면 힌트를 펼치세요.
4. 실습이 꼬이면 언제든 **리셋 셀**부터 다시 실행하면 처음 상태로 돌아갑니다.

> 데이터를 **바꾸는** 문제(1~4 · 24 · 25번)는 모두 여러분이 새로 만드는 `club` 표에서만 합니다. 그래서 조회 문제의 답은 앞 문제를 어떻게 풀었든 달라지지 않습니다.

> ⚠️ **리셋 셀을 다시 실행하면 여러분이 만든 `club` 표도 함께 사라집니다.** 24·25번을 풀던 중 리셋했다면 1~4번을 다시 실행한 뒤 이어 가세요.

## 실습 데이터 — 학사 DB

| 표 | 무엇 | 행 수 | 주요 열 |
|---|---|---|---|
| `department` | 학과 | 4 | `dept_id`(PK) · `dept_name` · `building` |
| `student` | 학생 | 32 | `stu_no`(PK) · `stu_name` · `dept_id`(FK) · `grade` · `class` · `gender` · `height` · `weight` |
| `subject` | 과목 | 10 | `sub_no`(PK) · `sub_name` · `credit` · `dept_id`(FK) |
| `enrol` | 수강 | 114 | **`(sub_no, stu_no)` 복합 기본키** · `sub_no`(FK) · `stu_no`(FK) · `enr_grade` |

`enrol` 은 학생과 과목을 잇는 **중간표**입니다 — 한 학생이 과목 여럿을, 한 과목에 학생 여럿 (M:N).

In [ ]:
# [제공 코드] — sqlite 실습 준비 (내용은 이해하지 않아도 됩니다 — 실행만 하세요)
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path(".") if Path("data").is_dir() else Path("..")
DB_PATH = ROOT / "output" / "school.db"
DB_PATH.parent.mkdir(exist_ok=True)

_conn = None


def get_conn():
    """실습용 sqlite 연결을 하나만 만들어 계속 재사용합니다."""
    global _conn
    if _conn is None:
        _conn = sqlite3.connect(DB_PATH, isolation_level=None)  # 실행하는 즉시 저장(자동 커밋)
        _conn.execute("pragma foreign_keys = on")               # 외래키 검사를 켭니다
    return _conn


def reset_db(script=None):
    """실습 DB를 처음 상태로 되돌립니다. 언제 몇 번을 다시 실행해도 안전합니다."""
    global _conn
    if _conn is not None:
        _conn.close()
        _conn = None
    DB_PATH.unlink(missing_ok=True)
    conn = get_conn()
    if script is not None:
        conn.executescript((ROOT / "data" / script).read_text(encoding="utf-8"))
        conn.execute("pragma foreign_keys = on")
    return conn


def run_sql(sql):
    """결과가 없는 SQL(CREATE·INSERT·UPDATE·DELETE 등)을 실행합니다."""
    get_conn().execute(sql)


def run_query(sql):
    """SELECT 결과를 pandas DataFrame 으로 돌려줍니다."""
    cur = get_conn().execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print("sqlite 준비 완료 —", DB_PATH)

In [ ]:
# [제공 코드] — 실습 테이블 전체 리셋 (언제든 다시 실행하면 처음 상태로 돌아갑니다)
reset_db("setup_school.sql")

for t in ["department", "student", "subject", "enrol"]:
    n = run_query(f"SELECT count(*) AS n FROM {t}")["n"][0]
    print(f"{t}: {n}행")

## 1. 표 만들기 (CREATE TABLE)

**배경**: 학교에 동아리 관리 기능을 붙입니다. 동아리 표를 새로 만드세요.

**요구사항**: `club` 이라는 표를 `STRICT` 로 만드세요. 열과 규칙은 아래와 같습니다.

| 열 | 타입 | 규칙 |
|---|---|---|
| `club_id` | integer | 기본키, 자동 증가 |
| `club_name` | text | 비어 있으면 안 되고, 중복도 안 됩니다 |
| `dept_id` | text | 비어 있으면 안 되고, `department` 의 `dept_id` 만 허용. 그리고 **학과가 지워지면 그 학과 동아리도 함께 지워지도록** 합니다 |
| `max_member` | int | 비어 있으면 안 되고, 1 이상이어야 합니다 |
| `room` | text | 비어 있어도 됩니다 |

**예시**

```
run_sql("""
CREATE TABLE 표이름 (
    열이름 타입 규칙,
    ...
) STRICT
""")
```

<details><summary>힌트</summary>

```text
접근방법:
- CREATE TABLE 뒤 괄호 안에 열을 한 줄씩 적는다
- 표 끝에 STRICT 를 붙여야 타입이 실제로 검사된다

세부구현:
1. 규칙 이름은 교안_01 의 제약조건 절 표에서 찾는다 (비어 있음·중복·범위·다른 표 참조)
2. 자동 증가 기본키는 타입을 integer 로 두고 열 뒤에 규칙 두 개를 잇는다
3. 다른 표의 값만 허용하는 규칙 뒤에, 부모가 지워질 때의 처리를 덧붙일 수 있다
4. 값의 범위를 거는 규칙은 괄호 안에 조건식을 받는다
```

</details>

In [ ]:
# 표를 만들 때 규칙(제약조건)을 함께 정한다 — STRICT 를 붙여야 타입이 실제로 검사된다
run_sql("""
CREATE TABLE club (
    club_id    integer PRIMARY KEY AUTOINCREMENT,
    club_name  text NOT NULL UNIQUE,
    dept_id    text NOT NULL REFERENCES department(dept_id) ON DELETE CASCADE,
    max_member int  NOT NULL CHECK (max_member >= 1),
    room       text
) STRICT
""")

<details><summary>해설</summary>

- `STRICT` 를 빠뜨리면 타입이 검사되지 않아 `max_member` 에 글자가 들어갈 수 있습니다.
- `REFERENCES department(dept_id)` 는 **존재하는 학과 코드**만 허용합니다. sqlite 에서는 `pragma foreign_keys = on` 이 켜져 있어야 실제로 막힙니다(준비 셀이 켜 둡니다).
- `ON DELETE CASCADE` 가 없으면 학과를 지우려 할 때 **삭제가 막힙니다**(자식이 남아 있으므로). 붙여 두면 부모와 함께 자식도 사라집니다.
- 흔한 실수: `CHECK (max_member > 0)` 도 정답입니다 — 정수라서 `>= 1` 과 같습니다. 반대로 **규칙을 엉뚱한 열에 거는 것**은 오답입니다 — 자가채점이 동작으로 확인합니다.

</details>

In [ ]:
# [자가채점]
cols = run_query("pragma table_info(club)")
assert list(cols["name"]) == ["club_id", "club_name", "dept_id", "max_member", "room"], \
    f"열 이름·순서를 확인하세요: {list(cols['name'])}"

ddl = run_query("SELECT sql FROM sqlite_master WHERE name = 'club'")["sql"][0].lower()
assert "strict" in ddl, "표 끝에 STRICT 를 붙이세요"
assert "autoincrement" in ddl, "club_id 는 자동 증가 기본키여야 합니다"

assert run_query("pragma foreign_keys").iloc[0, 0] == 1, \
    "외래키 검사가 꺼져 있습니다 — 맨 위 준비 셀을 다시 실행하세요"

# 규칙을 '어디에' 걸었는지는 동작으로 확인합니다. 검사용 행은 마지막에 되돌립니다.
run_sql("BEGIN")
try:
    def blocked(sql):
        try:
            run_sql(sql)
            return False
        except sqlite3.IntegrityError:
            return True

    # 검사마다 이름·학과를 모두 다르게 둡니다 — 그래야 '어느 규칙이 막았는지'가 분명해집니다.
    run_sql("INSERT INTO club (club_name, dept_id, max_member) VALUES ('검사용', 'CS', 10)")
    assert blocked("INSERT INTO club (club_name, dept_id, max_member) "
                   "VALUES ('검사용', 'ME', 10)"), \
        "같은 동아리 이름이 두 번 들어갑니다 — unique 를 club_name 에 거세요"
    assert blocked("INSERT INTO club (club_name, dept_id, max_member) "
                   "VALUES ('검사용2', 'XX', 10)"), \
        "없는 학과 코드가 들어갑니다 — dept_id 에 외래키를 거세요"
    assert blocked("INSERT INTO club (club_name, dept_id, max_member) "
                   "VALUES ('검사용3', 'EE', 0)"), \
        "정원 0 이 들어갑니다 — max_member 에 check 를 거세요"

    run_sql("INSERT INTO department (dept_id, dept_name) VALUES ('AI', '인공지능')")
    run_sql("INSERT INTO club (club_name, dept_id, max_member) VALUES ('검사용4', 'AI', 10)")
    try:
        run_sql("DELETE FROM department WHERE dept_id = 'AI'")
    except sqlite3.IntegrityError:
        raise AssertionError("학과를 지우려니 외래키가 막습니다 — "
                             "dept_id 외래키에 on delete cascade 를 붙이세요")
    left = run_query("SELECT count(*) AS n FROM club WHERE dept_id = 'AI'")["n"][0]
    assert left == 0, "학과를 지웠는데 동아리가 남았습니다 — on delete cascade 를 확인하세요"
finally:
    run_sql("ROLLBACK")

print("✅ 통과!")

## 2. 행 넣기 (INSERT — 값 건네기와 RETURNING)

**배경**: 만든 동아리 표에 첫 동아리를 등록합니다. 그런데 동아리방 이름에 **작은따옴표가 들어 있습니다.**

**요구사항**: `club` 에 아래 한 행을 넣으세요. `club_id` 는 자동 증가이므로 적지 않습니다.

| club_name | dept_id | max_member | room |
|---|---|---|---|
| 알고리즘연구회 | CS | 20 | 공학1관 'A'동 |

- `room` 값에 작은따옴표가 있어 **문장에 그대로 붙이면 SQL 이 중간에서 끊깁니다.** 값을 문장에 붙이지 말고 **물음표 자리에 따로 건네** 넣으세요(교안에서 본 방법입니다).
- 넣은 뒤 **새로 생긴 `club_id` 를 그 자리에서 돌려받으세요.** 실행한 커서는 `cur` 에, 돌려받은 값은 `new_id` 에 담고 출력합니다.

**예시**

```
cur = get_conn().execute("INSERT INTO 표 (열1, 열2) VALUES (?, ?) RETURNING 열", (값1, 값2))
```

<details><summary>힌트</summary>

```text
접근방법:
- 값을 문장에 붙이지 않고 따로 건네면 따옴표가 들어 있어도 문장이 끊기지 않는다
- 넣은 행을 그 자리에서 돌려받는 절이 있다

세부구현:
1. 자동 증가 기본키는 열 목록에서 뺀다
2. 값 자리마다 물음표를 놓고, 값들은 튜플로 함께 넘긴다
3. 돌려받을 열을 지정하는 절을 문장 끝에 붙인다
4. 커서에서 한 행을 꺼내 첫 값을 new_id 에 담는다
```

</details>

In [ ]:
# 값에 따옴표가 있어 문장에 붙이면 SQL 이 끊긴다 — 물음표 자리에 값을 따로 건넨다
# RETURNING 은 방금 만들어진 행을 그 자리에서 돌려준다(자동 증가 id 를 확인할 때 요긴하다)
cur = get_conn().execute(
    "INSERT INTO club (club_name, dept_id, max_member, room) "
    "VALUES (?, ?, ?, ?) RETURNING club_id",
    ("알고리즘연구회", "CS", 20, "공학1관 'A'동"),
)
new_id = cur.fetchone()[0]
print("새로 생긴 club_id:", new_id)
display(run_query("SELECT * FROM club"))

<details><summary>해설</summary>

- 값을 문장에 이어 붙이면 `'공학1관 'A'동'` 이 되어 따옴표 짝이 어긋납니다 — `OperationalError: near "A": syntax error`. 물음표(`?`) 자리에 값을 **따로 건네면** 데이터베이스가 값으로만 취급해 이런 일이 없습니다.
- 작은따옴표를 두 번(`''`) 적어 escape 해도 이 한 줄은 들어갑니다. 하지만 값이 어디서 오는지 모르는 상황(사용자 입력·파일)에서는 그런 처리를 **빠뜨리는 순간** 문장이 깨지거나 남이 넣은 글자가 **문장으로 해석되는 사고**(SQL 주입)가 납니다. 그래서 실무에서는 값을 붙이지 않고 **따로 건네는 쪽을 기본**으로 씁니다.
- `RETURNING club_id` 는 방금 생긴 값을 그 자리에서 돌려줍니다. 이것이 없으면 `SELECT` 를 한 번 더 해야 하고, 그 사이 다른 사람이 넣은 행과 헷갈릴 수 있습니다.
- `run_sql()` 은 값을 따로 받지 못합니다. 그래서 이 문제만 `get_conn().execute(...)` 를 직접 씁니다 — 교안에서 쓴 그 모양입니다.

</details>

In [ ]:
# [자가채점]
row = run_query("SELECT * FROM club WHERE club_name = '알고리즘연구회'")
assert len(row) == 1, "알고리즘연구회 가 정확히 한 행 있어야 합니다"
assert row["dept_id"][0] == "CS", "dept_id 는 'CS' 여야 합니다"
assert int(row["max_member"][0]) == 20, "max_member 는 20 이어야 합니다"
assert row["room"][0] == "공학1관 'A'동", \
    f"room 은 작은따옴표까지 그대로 들어가야 합니다: {row['room'][0]}"
assert int(new_id) == 1, \
    "RETURNING 으로 돌려받은 club_id 를 new_id 에 담으세요 (첫 행이라 1 입니다)"
assert cur.description is not None and cur.description[0][0] == "club_id", \
    "cur 는 INSERT ... RETURNING club_id 를 실행한 커서여야 합니다 "
print("✅ 통과!")

## 3. 일부 열만 넣기 (INSERT — 일부 열)

**배경**: 아직 동아리방이 배정되지 않은 동아리도 미리 등록해 둡니다.

**요구사항**: `club` 에 아래 두 동아리를 넣되, **`room` 은 적지 마세요**. 한 번의 `INSERT` 로 두 행을 함께 넣습니다.

| club_name | dept_id | max_member |
|---|---|---|
| 로봇제작반 | ME | 15 |
| 데이터분석회 | DS | 25 |

**예시**

```
run_sql("INSERT INTO 표 (열1, 열2) VALUES (값1, 값2), (값3, 값4)")
```

<details><summary>힌트</summary>

```text
접근방법:
- 넣지 않을 열은 열 목록에서 아예 빼면 된다
- VALUES 뒤에 괄호 묶음을 콤마로 이으면 여러 행이 한 번에 들어간다

세부구현:
1. 열 목록에 room 을 넣지 않는다
2. 값 묶음 두 개를 콤마로 잇는다
3. 넣은 뒤 조회해 room 이 비어 있는지 확인한다
```

</details>

In [ ]:
# 열 목록에서 뺀 room 은 NULL 이 된다 — 뺄 수 있는 것은 NOT NULL 이 아닌 열뿐이다
run_sql("""
INSERT INTO club (club_name, dept_id, max_member) VALUES
    ('로봇제작반', 'ME', 15),
    ('데이터분석회', 'DS', 25)
""")
display(run_query("SELECT * FROM club ORDER BY club_id"))

<details><summary>해설</summary>

- 적지 않은 `room` 은 `NULL` 이 됩니다. `NOT NULL` 인 열을 빼면 거부되니, **빼도 되는 열은 `NOT NULL` 이 아닌 열**뿐입니다.
- 흔한 실수: `VALUES ('로봇제작반', 'ME', 15, NULL)` 처럼 `NULL` 을 직접 적는 것도 결과는 같지만, 열 목록에서 빼는 쪽이 의도가 분명합니다.

</details>

In [ ]:
# [자가채점]
rows = run_query("SELECT club_name, dept_id, max_member, room FROM club ORDER BY club_id")
assert len(rows) == 3, f"club 은 3행이어야 합니다 (지금 {len(rows)}행)"
target = rows[rows["club_name"].isin(["로봇제작반", "데이터분석회"])]
assert len(target) == 2, "로봇제작반·데이터분석회 두 행이 있어야 합니다"
assert target["room"].isna().all(), "두 동아리의 room 은 비어 있어야(NULL) 합니다"
assert sorted(target["max_member"].tolist()) == [15, 25], "정원을 확인하세요"
print("✅ 통과!")

## 4. 표에 열 붙이기 (ALTER TABLE)

**배경**: 동아리마다 창립 연도를 함께 기록하기로 했습니다. 표는 이미 데이터가 들어 있으니 다시 만들 수는 없습니다.

**요구사항**: `club` 표에 `founded_year` 열을 붙이세요. 타입은 `int` 이고, **비어 있으면 안 되며 기본값은 2026** 입니다.

**예시**

```
run_sql("ALTER TABLE 표 ADD COLUMN 열이름 타입 규칙")
```

<details><summary>힌트</summary>

```text
접근방법:
- 이미 들어 있는 행의 새 열은 무엇으로 채울지 정해 줘야 한다

세부구현:
1. ALTER TABLE 로 열을 추가한다
2. 비어 있으면 안 되는 열이므로 기본값을 함께 준다
3. 조회해 기존 3행이 모두 같은 값으로 채워졌는지 확인한다
```

</details>

In [ ]:
# 이미 행이 있는 표에 NOT NULL 열을 붙이려면 DEFAULT 로 채울 값을 함께 줘야 한다
run_sql("ALTER TABLE club ADD COLUMN founded_year int NOT NULL DEFAULT 2026")
display(run_query("SELECT * FROM club ORDER BY club_id"))

<details><summary>해설</summary>

- `DEFAULT` 없이 `NOT NULL` 만 주면 기존 행을 `NULL` 로 채울 수 없어 **실행이 거부**됩니다.
- sqlite 는 `ALTER TABLE` 로 **제약조건을 새로 붙이지는 못합니다.** 그래서 표를 만들 때 규칙을 다 정해 두는 것이 중요합니다.

</details>

In [ ]:
# [자가채점]
cols = run_query("pragma table_info(club)")
assert "founded_year" in list(cols["name"]), "founded_year 열이 없습니다"
vals = run_query("SELECT founded_year FROM club")["founded_year"].tolist()
assert vals == [2026, 2026, 2026], f"기존 3행이 모두 2026 이어야 합니다: {vals}"
row = cols[cols["name"] == "founded_year"].iloc[0]
assert int(row["notnull"]) == 1, "founded_year 에 NOT NULL 이 빠졌습니다"
assert str(row["dflt_value"]).strip() == "2026", "기본값 2026 을 함께 주세요"
print("✅ 통과!")

## 5. 전체 조회 (SELECT *)

**배경**: 이제 조회입니다. 먼저 학생 명단을 통째로 봅니다.

**요구사항**: `student` 표의 **모든 열·모든 행**을 조회해 `q5` 라는 이름에 담으세요.

**예시**: 결과는 32행 8열입니다.

```
q5 = run_query("SELECT ... FROM ...")
display(q5)
```

<details><summary>힌트</summary>

```text
접근방법:
- 모든 열을 뜻하는 기호가 하나 있다

세부구현:
1. SELECT 뒤에 모든 열 기호를 적는다
2. FROM 뒤에 표 이름을 적는다
3. 결과를 q5 에 담고 display 로 확인한다
```

</details>

In [ ]:
# * 는 모든 열이라는 뜻 — 실무에서는 필요한 열만 적는 편이 빠르고 읽기 좋다
q5 = run_query("SELECT * FROM student")
display(q5)

<details><summary>해설</summary>

- `*` 는 모든 열입니다. 실무에서는 필요한 열만 적는 편이 빠르고 읽기 좋습니다.
- 행이 많을 때는 `q5.head()` 로 앞부분만 봐도 됩니다.

</details>

In [ ]:
# [자가채점]
assert len(q5) == 32, f"32행이어야 합니다 (지금 {len(q5)}행)"
assert list(q5.columns) == ["stu_no", "stu_name", "dept_id", "grade",
                            "class", "gender", "height", "weight"], \
    f"모든 열이 나와야 합니다: {list(q5.columns)}"
print("✅ 통과!")

## 6. 필요한 열만 조회 (SELECT 열)

**배경**: 과목 목록에서 이름과 학점만 있으면 되는 상황입니다.

**요구사항**: `subject` 표에서 **`sub_name` 과 `credit` 두 열만** 조회해 `q6` 에 담으세요. 열 순서도 `sub_name` · `credit` 입니다.

**예시**: 결과는 10행 2열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- SELECT 뒤에 볼 열만 콤마로 나열한다

세부구현:
1. 두 열 이름을 순서대로 적는다
2. 결과를 q6 에 담는다
```

</details>

In [ ]:
# SELECT 는 볼 열만 고른다 — 행 수는 그대로다
q6 = run_query("SELECT sub_name, credit FROM subject")
display(q6)

<details><summary>해설</summary>

- `SELECT` 는 **어떤 열을 볼지**만 정합니다. 행 수는 그대로입니다.
- 열 순서는 적은 순서 그대로 나옵니다.

</details>

In [ ]:
# [자가채점]
assert len(q6) == 10, "10행이어야 합니다"
assert list(q6.columns) == ["sub_name", "credit"], f"열은 sub_name·credit 입니다: {list(q6.columns)}"
want = run_query("SELECT sub_name, credit FROM subject")
assert sorted(map(tuple, q6.values.tolist())) == sorted(map(tuple, want.values.tolist())), \
    "과목명과 학점 값이 subject 표와 다릅니다"
print("✅ 통과!")

## 7. 두 조건을 함께 (WHERE + AND)

**배경**: 신입생 여학생 대상 프로그램 안내를 보내려 합니다.

**요구사항**: `student` 에서 **1학년이면서 성별이 `'F'`** 인 학생의 `stu_no` · `stu_name` 두 열을 조회해 `q7` 에 담으세요.

**예시**: 결과는 5행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 조건이 동시에 참이어야 하므로 AND 로 잇는다

세부구현:
1. WHERE 뒤에 학년 조건을 적는다
2. AND 로 성별 조건을 잇는다 (글자 값은 작은따옴표)
```

</details>

In [ ]:
# AND 는 두 조건이 모두 참인 행만 남긴다 (글자 값은 작은따옴표로 감싼다)
q7 = run_query("""
SELECT stu_no, stu_name
FROM student
WHERE grade = 1 AND gender = 'F'
""")
display(q7)

<details><summary>해설</summary>

- `AND` 는 **두 조건이 모두 참**인 행만 남깁니다.
- 흔한 실수: `gender = F` 처럼 따옴표를 빼면 `F` 라는 **열 이름**으로 읽혀 오류가 납니다.

</details>

In [ ]:
# [자가채점]
assert len(q7) == 5, f"5행이어야 합니다 (지금 {len(q7)}행)"
assert list(q7.columns) == ["stu_no", "stu_name"], f"열은 stu_no·stu_name 입니다: {list(q7.columns)}"
assert set(q7["stu_name"]) == set(['서다인', '심수정', '오예은', '이태연', '정다인']), \
    f"1학년 여학생이 아닌 학생이 섞여 있습니다: {sorted(q7['stu_name'])}"
print("✅ 통과!")

## 8. 둘 중 하나면 (WHERE + OR)

**배경**: 졸업 예정자 간담회에 **4학년 전원**과 **데이터과학과(`DS`) 학생 전원**을 부릅니다.

**요구사항**: `student` 에서 **4학년이거나 학과가 `'DS'`** 인 학생의 `stu_no` · `stu_name` · `grade` · `dept_id` 를 조회해 `q8` 에 담으세요.

**예시**: 결과는 10행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 둘 중 하나만 참이어도 되므로 OR 로 잇는다

세부구현:
1. WHERE 뒤에 학년 조건을 적는다
2. OR 로 학과 조건을 잇는다
3. 4학년이면서 DS 인 학생은 한 번만 나온다는 것을 확인한다
```

</details>

In [ ]:
# OR 는 둘 중 하나만 참이어도 남긴다 — 둘 다 만족하는 학생도 한 번만 나온다
q8 = run_query("""
SELECT stu_no, stu_name, grade, dept_id
FROM student
WHERE grade = 4 OR dept_id = 'DS'
""")
display(q8)

<details><summary>해설</summary>

- `OR` 는 **둘 중 하나라도 참**이면 남깁니다. 두 조건을 모두 만족하는 학생도 결과에는 **한 번만** 나옵니다.
- `AND` 와 `OR` 를 섞을 때는 반드시 괄호로 묶으세요 — `AND` 가 먼저 계산됩니다.

</details>

In [ ]:
# [자가채점]
assert len(q8) == 10, f"10행이어야 합니다 (지금 {len(q8)}행)"
assert list(q8.columns) == ["stu_no", "stu_name", "grade", "dept_id"], \
    f"열 이름·순서를 확인하세요: {list(q8.columns)}"
assert ((q8["grade"] == 4) | (q8["dept_id"] == "DS")).all(), "조건에 맞지 않는 행이 있습니다"
print("✅ 통과!")

## 9. 범위로 뽑기 (BETWEEN)

**배경**: 단체복 M 사이즈 대상자를 추립니다.

**요구사항**: `student` 에서 **키가 162 이상 172 이하**인 학생의 `stu_name` · `height` 를 조회해 `q9` 에 담으세요. `BETWEEN` 을 쓰세요.

**예시**: 결과는 13행입니다. **양끝인 162 와 172 도 포함**합니다 — 키가 정확히 162.0 인 학생(이태연)과 172.0 인 학생(옥성우)이 실제로 있어, 양끝을 빼면 결과가 달라집니다.

<details><summary>힌트</summary>

```text
접근방법:
- BETWEEN 은 양끝을 포함하는 범위 조건이다

세부구현:
1. WHERE 뒤에 열 이름을 적고 BETWEEN 으로 두 값을 잇는다
2. 양끝 값이 결과에 들어오는지 눈으로 확인한다
3. 키가 비어 있는 학생은 자동으로 빠진다는 점을 확인한다
```

</details>

In [ ]:
# BETWEEN 은 양끝을 포함한다. 키가 NULL 인 학생은 어떤 비교에도 참이 아니라 자동으로 빠진다
q9 = run_query("""
SELECT stu_name, height
FROM student
WHERE height BETWEEN 162 AND 172
""")
display(q9)

<details><summary>해설</summary>

- `BETWEEN a AND b` 는 **양끝을 포함**합니다. `height >= 162 AND height <= 172` 와 같습니다.
- `>` 와 `<` 로 바꿔 쓰면 양끝의 두 학생(이태연 162.0 · 옥성우 172.0)이 빠져 결과가 2행 줄어듭니다 — 경계를 포함하는지가 실제로 답을 가릅니다.
- 키가 `NULL` 인 학생은 어떤 비교에도 참이 되지 않아 자동으로 빠집니다.

</details>

In [ ]:
# [자가채점]
assert len(q9) == 13, f"13행이어야 합니다 (지금 {len(q9)}행)"
assert list(q9.columns) == ["stu_name", "height"], f"열은 stu_name·height 입니다: {list(q9.columns)}"
assert q9["height"].between(162, 172).all(), "범위를 벗어난 행이 있습니다"
assert "이태연" in set(q9["stu_name"]) and "옥성우" in set(q9["stu_name"]), \
    "양끝 값(162·172)인 이태연·옥성우 이 빠졌습니다 — BETWEEN 은 양끝을 포함합니다"
print("✅ 통과!")

## 10. 목록으로 뽑기 (IN)

**배경**: 컴퓨터정보(`CS`)와 데이터과학(`DS`) 두 학과 학생에게 특강을 안내합니다.

**요구사항**: `student` 에서 학과가 **`'CS'` 또는 `'DS'`** 인 학생의 `stu_no` · `stu_name` · `dept_id` 를 조회해 `q10` 에 담으세요. `IN` 을 쓰세요.

**예시**: 결과는 16행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 값이 목록 안에 있는지 묻는 조건을 쓴다

세부구현:
1. WHERE 뒤에 열 이름을 적고 IN 뒤 괄호에 값들을 콤마로 나열한다
2. 글자 값이므로 각각 작은따옴표로 감싼다
```

</details>

In [ ]:
# IN 은 = 'CS' OR = 'DS' 의 줄임 — 값이 늘어날수록 읽기 좋아진다
q10 = run_query("""
SELECT stu_no, stu_name, dept_id
FROM student
WHERE dept_id IN ('CS', 'DS')
""")
display(q10)

<details><summary>해설</summary>

- `IN ('CS', 'DS')` 는 `dept_id = 'CS' OR dept_id = 'DS'` 와 같습니다. 값이 늘어날수록 `IN` 이 훨씬 읽기 좋습니다.
- 반대는 `NOT IN (...)` 입니다.

</details>

In [ ]:
# [자가채점]
assert len(q10) == 16, f"16행이어야 합니다 (지금 {len(q10)}행)"
assert set(q10["dept_id"]) == {"CS", "DS"}, "CS·DS 학생만 나와야 합니다"
print("✅ 통과!")

## 11. 패턴으로 찾기 (LIKE — 앞이 같은)

**배경**: 성이 '김'인 학생을 찾습니다.

**요구사항**: `student` 에서 **이름이 `'김'` 으로 시작**하는 학생의 `stu_no` · `stu_name` 을 조회해 `q11` 에 담으세요.

**예시**: 결과는 4행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- LIKE 패턴에서 아무 글자 여러 개를 뜻하는 기호를 쓴다

세부구현:
1. WHERE 뒤에 열 이름과 LIKE 를 적는다
2. 시작 글자 뒤에 여러 글자 기호를 붙인다
```

</details>

In [ ]:
# % 는 아무 글자 0개 이상 — '김%' 는 김으로 시작하는 이름
q11 = run_query("""
SELECT stu_no, stu_name
FROM student
WHERE stu_name LIKE '김%'
""")
display(q11)

<details><summary>해설</summary>

- `%` 는 **아무 글자 0개 이상**, `_` 는 **정확히 한 글자**입니다.
- `'김%'` 는 김으로 시작, `'%김'` 은 김으로 끝, `'%김%'` 은 김이 들어간 모든 이름입니다.

</details>

In [ ]:
# [자가채점]
assert len(q11) == 4, f"4행이어야 합니다 (지금 {len(q11)}행)"
assert q11["stu_name"].str.startswith("김").all(), "김으로 시작하지 않는 이름이 있습니다"
print("✅ 통과!")

## 12. 패턴 위치 바꾸기 (LIKE — 끝이 같은)

**배경**: 이번엔 끝 글자로 찾습니다.

**요구사항**: `student` 에서 **이름이 `'우'` 로 끝나는** 학생의 `stu_no` · `stu_name` 을 조회해 `q12` 에 담으세요.

**예시**: 결과는 7행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 앞 문제와 같은 기호를 반대편에 붙인다

세부구현:
1. 여러 글자 기호를 앞에 두고 끝 글자를 뒤에 적는다
```

</details>

In [ ]:
# 기호의 위치가 뜻을 바꾼다 — '%우' 는 우로 끝나는 이름
q12 = run_query("""
SELECT stu_no, stu_name
FROM student
WHERE stu_name LIKE '%우'
""")
display(q12)

<details><summary>해설</summary>

- 기호의 **위치**가 의미를 바꿉니다. `'%우'` 는 끝이 우, `'우%'` 는 시작이 우입니다.
- 이름 가운데에 있는 글자를 찾으려면 양쪽에 `%` 를 붙입니다.

</details>

In [ ]:
# [자가채점]
assert len(q12) == 7, f"7행이어야 합니다 (지금 {len(q12)}행)"
assert q12["stu_name"].str.endswith("우").all(), "우로 끝나지 않는 이름이 있습니다"
print("✅ 통과!")

## 13. 빈 값 찾기, 그리고 그 반대 (IS NULL · IS NOT NULL)

**배경**: 신체검사 기록이 빠진 학생에게는 재검을 안내하고, 기록이 있는 학생만 따로 통계에 씁니다. **두 명단이 다 필요합니다.**

**요구사항**: `student` 에서 `stu_no` · `stu_name` 을 조회해 두 결과를 만드세요.

1. 키(`height`)가 **비어 있는** 학생 → `q13` (3행)
2. 키가 **기록된** 학생 → `q13b` (29행)

**예시**: 두 결과의 행 수를 더하면 전체 학생 수 32 가 됩니다 — 빠짐도 겹침도 없어야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 빈 값은 = 로 비교되지 않는다. 전용 조건이 따로 있다
- 그 조건을 뒤집는 말이 조건 안에 들어간다 (조건 전체를 NOT 으로 감싸는 것이 아니다)

세부구현:
1. WHERE 뒤에 열 이름을 적고 빈 값 전용 조건을 붙인다
2. 같은 질의에서 조건만 뒤집어 두 번째 결과를 만든다
```

</details>

In [ ]:
# NULL 은 = 로 비교되지 않는다 — 빈 값 전용 조건인 IS NULL 을 쓴다
q13 = run_query("""
SELECT stu_no, stu_name
FROM student
WHERE height IS NULL
""")
display(q13)

# 반대 조건은 IS NOT NULL — 두 결과를 합치면 전체가 되고, 겹치는 학생은 없다
q13b = run_query("""
SELECT stu_no, stu_name
FROM student
WHERE height IS NOT NULL
""")
display(q13b)

<details><summary>해설</summary>

- `NULL` 은 '값이 없다'는 표시라 **무엇과도 같다고 판정되지 않습니다.** `= NULL` 로는 한 행도 못 찾고, `!= NULL` 로도 못 찾습니다 — 그래서 전용 조건이 있습니다.
- `NOT (height IS NULL)` 이라고 써도 결과는 같지만, `IS NOT NULL` 이 관례이고 읽기도 좋습니다.
- 두 결과가 3 + 29 = 32 로 딱 나뉘면 `NULL` 을 빠짐없이 갈랐다는 뜻입니다.
- `WHERE height > 0` 으로 대신해도 이 데이터에서는 같은 29행이 나옵니다(기록된 키가 모두 양수라서). 하지만 그건 **우연**입니다 — 키가 `0` 으로 잘못 기록된 학생이 생기면 그 학생은 어느 명단에도 안 들어가 조용히 사라집니다. **'값이 있느냐'를 묻고 싶으면 값을 비교하지 말고 `IS NOT NULL` 로 물어야** 합니다.

</details>

In [ ]:
# [자가채점]
assert len(q13) == 3, f"q13 은 3행이어야 합니다 (지금 {len(q13)}행)"
assert len(q13b) == 29, \
    f"q13b 는 29행이어야 합니다 (지금 {len(q13b)}행)"
assert sorted(q13["stu_name"]) == ['김종헌', '박희철', '한소민'], \
    f"키가 비어 있는 학생의 이름을 확인하세요: {sorted(q13['stu_name'])}"
for name, df in [("q13", q13), ("q13b", q13b)]:
    assert list(df.columns) == ["stu_no", "stu_name"], \
        f"{name} 의 열은 stu_no·stu_name 입니다: {list(df.columns)}"
overlap = set(q13["stu_no"]) & set(q13b["stu_no"])
assert not overlap, f"두 명단에 같은 학생이 들어 있습니다: {sorted(overlap)}"
assert len(q13) + len(q13b) == 32, \
    "두 명단을 합치면 전체 학생 32명이어야 합니다 — 빠진 학생이 있습니다"
print("✅ 통과!")

## 14. 중복 없이 종류만 (DISTINCT)

**배경**: 학생이 실제로 속해 있는 학과가 몇 가지인지 봅니다.

**요구사항**: `student` 표에 등장하는 **학과 코드의 종류**를 중복 없이 조회해 `q14` 에 담으세요. 열은 `dept_id` 하나입니다.

**예시**: 결과는 4행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 중복을 없애는 키워드를 SELECT 바로 뒤에 붙인다

세부구현:
1. SELECT 와 열 이름 사이에 그 키워드를 넣는다
```

</details>

In [ ]:
# DISTINCT 는 결과 행 전체를 기준으로 중복을 없앤다
q14 = run_query("SELECT DISTINCT dept_id FROM student")
display(q14)

<details><summary>해설</summary>

- `DISTINCT` 는 **결과 행 전체**를 기준으로 중복을 없앱니다. 열을 두 개 적으면 '두 값의 조합'이 같은 행만 합쳐집니다.
- 값이 비어 있는(`NULL`) 경우도 하나의 종류로 함께 나옵니다.

</details>

In [ ]:
# [자가채점]
assert len(q14) == 4, f"4행이어야 합니다 (지금 {len(q14)}행)"
assert list(q14.columns) == ["dept_id"], f"열은 dept_id 하나입니다: {list(q14.columns)}"
assert sorted(q14["dept_id"]) == ['CS', 'DS', 'EE', 'ME'], \
    f"학과 코드를 확인하세요 — 중복 없이 4가지입니다: {sorted(q14['dept_id'])}"
print("✅ 통과!")

## 15. 종류만 추리기 (DISTINCT — 다른 표)

**배경**: 개설 과목의 학점이 몇 종류인지 봅니다.

**요구사항**: `subject` 표에 등장하는 **학점의 종류**를 중복 없이 조회해 `q15` 에 담으세요. 열은 `credit` 하나입니다.

**예시**: 결과는 2행입니다 — 개설 과목의 학점은 2가지뿐입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 14번과 같은 키워드를 다른 표에 적용한다

세부구현:
1. DISTINCT 로 종류만 남긴다
```

</details>

In [ ]:
# 같은 DISTINCT 를 다른 표에 적용한다 — 학점이 몇 가지인지 한눈에 보인다
q15 = run_query("SELECT DISTINCT credit FROM subject")
display(q15)

<details><summary>해설</summary>

- `DISTINCT` 는 표가 달라져도 하는 일이 같습니다 — 결과 행에서 중복을 없앱니다.
- 순서가 필요하면 `ORDER BY` 를 붙일 수 있습니다(16·17번에서 본격적으로 다룹니다). 여기서는 **어떤 값들이 있는지**만 봅니다.

</details>

In [ ]:
# [자가채점]
assert len(q15) == 2, f"2행이어야 합니다 (지금 {len(q15)}행)"
assert list(q15.columns) == ["credit"], f"열은 credit 하나입니다: {list(q15.columns)}"
assert sorted(q15["credit"]) == [2, 3], \
    f"학점의 종류를 확인하세요: {sorted(q15['credit'])}"
print("✅ 통과!")

## 16. 여러 기준으로 정렬 (ORDER BY 다중 열)

**배경**: 학년별로 묶어 보되, 같은 학년 안에서는 키가 큰 학생부터 보고 싶습니다.

**요구사항**: `student` 를 **학년 오름차순**으로 세우고, 같은 학년 안에서는 **키 내림차순**으로 정렬해 `stu_name` · `grade` · `height` 를 조회하고 `q16` 에 담으세요.

**예시**: 결과는 32행이고, 첫 행은 **조민우** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- ORDER BY 뒤에 기준을 콤마로 여러 개 적으면 앞 기준이 우선한다

세부구현:
1. 첫 기준은 학년 오름차순
2. 둘째 기준은 키 내림차순 (내림차순 키워드를 붙인다)
```

</details>

In [ ]:
# 앞 기준이 우선하고 값이 같을 때만 뒤 기준을 본다. DESC 는 그 기준 하나에만 붙는다
q16 = run_query("""
SELECT stu_name, grade, height
FROM student
ORDER BY grade, height DESC
""")
display(q16)

<details><summary>해설</summary>

- 기준이 여러 개면 **앞 기준이 우선**하고, 값이 같을 때만 뒤 기준을 봅니다.
- `DESC` 는 그 기준 하나에만 붙습니다. `ORDER BY grade DESC, height DESC` 처럼 둘 다 내림차순이라면 각각 적어야 합니다.

</details>

In [ ]:
# [자가채점]
assert len(q16) == 32, "32행이어야 합니다"
assert q16["stu_name"][0] == "조민우", \
    f"첫 행은 조민우 여야 합니다 (지금 {q16['stu_name'][0]})"
assert q16["grade"].tolist() == sorted(q16["grade"].tolist()), "학년이 오름차순이어야 합니다"
print("✅ 통과!")

## 17. 다음 페이지 보기 (ORDER BY + LIMIT + OFFSET)

**배경**: 과목 목록을 한 페이지에 3개씩 보여 줍니다. 지금 **두 번째 페이지**(4~6위)를 만듭니다.

**요구사항**: `subject` 를 **학점 내림차순**, 같으면 **과목명 오름차순**으로 정렬한 뒤 **앞의 3개를 건너뛰고 그다음 3개**를 조회해 `q17` 에 담으세요. 열은 `sub_name` · `credit` 입니다.

**예시**: 결과는 3행이고 첫 행은 **인공지능개론** 입니다 (1~3위는 기계설계 로 시작하는 앞 페이지입니다).

<details><summary>힌트</summary>

```text
접근방법:
- 줄을 세운 다음, 앞의 몇 개를 건너뛰고 그다음 몇 개를 가져온다
- 건너뛰는 개수를 지정하는 키워드가 따로 있다

세부구현:
1. ORDER BY 로 두 기준을 적는다
2. 가져올 개수를 정한다
3. 건너뛸 개수를 지정하는 키워드를 덧붙인다
```

</details>

In [ ]:
# 정렬한 뒤 앞의 3개를 건너뛰고(OFFSET) 그다음 3개를 가져온다(LIMIT)
q17 = run_query("""
SELECT sub_name, credit
FROM subject
ORDER BY credit DESC, sub_name
LIMIT 3 OFFSET 3
""")
display(q17)

<details><summary>해설</summary>

- `LIMIT` 은 **정렬한 뒤** 잘라 냅니다. `ORDER BY` 없이 쓰면 어떤 행이 나올지 보장되지 않습니다.
- `OFFSET m` 은 앞의 m 개를 건너뜁니다. 한 페이지에 n 개씩 보여 줄 때 p 번째 페이지는 `LIMIT n OFFSET (p-1)*n` 입니다.
- 건너뛴 뒤 남은 행이 3개보다 적으면 그만큼만 나옵니다.

</details>

In [ ]:
# [자가채점]
assert len(q17) == 3, f"3행이어야 합니다 (지금 {len(q17)}행)"
assert q17["sub_name"].tolist() == ['인공지능개론', '자료구조', '전자기학'], \
    f"4~6위가 아닙니다 — 앞의 3개를 건너뛰었는지 확인하세요: {q17['sub_name'].tolist()}"
print("✅ 통과!")

## 18. 계산 열과 별칭 (AS)

**배경**: 한 학기는 16주입니다. 과목마다 학기 전체 수업 시간을 계산해 보여 줍니다.

**요구사항**: `subject` 에서 과목명과 **`credit` 에 16을 곱한 값**을 조회하되, 계산 결과의 이름을 **`total_hours`** 로 붙이세요. 결과를 `q18` 에 담습니다. 열 순서는 `sub_name` · `total_hours` 입니다.

**예시**: '데이터베이스'(3학점)의 `total_hours` 는 48 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- SELECT 의 열 자리에는 계산식도 올 수 있다
- 계산 결과에 이름을 붙이는 키워드가 있다

세부구현:
1. 열 자리에 곱셈식을 적는다
2. 그 뒤에 이름 붙이는 키워드와 원하는 이름을 적는다
```

</details>

In [ ]:
# 열 자리에는 계산식도 올 수 있다. AS 로 이름을 붙이지 않으면 식이 그대로 열 이름이 된다
q18 = run_query("""
SELECT sub_name,
       credit * 16 AS total_hours
FROM subject
""")
display(q18)

<details><summary>해설</summary>

- 별칭(`AS`)을 붙이지 않으면 결과 열 이름이 `credit * 16` 처럼 식 그대로 나옵니다.
- 별칭은 `SELECT` 를 처리할 때 비로소 생기므로 **`WHERE` 에서는 쓸 수 없습니다.** `ORDER BY` 에서는 쓸 수 있습니다.

</details>

In [ ]:
# [자가채점]
assert len(q18) == 10, "10행이어야 합니다"
assert list(q18.columns) == ["sub_name", "total_hours"], \
    f"열은 sub_name·total_hours 입니다: {list(q18.columns)}"
want = run_query("SELECT sub_name, credit * 16 AS total_hours FROM subject")
got = sorted(map(tuple, q18.values.tolist()))
assert got == sorted(map(tuple, want.values.tolist())), \
    "모든 과목의 total_hours 가 credit * 16 이어야 합니다 (한 과목만 우연히 맞은 것은 아닌지 확인하세요)"
print("✅ 통과!")

## 19. 세는 두 가지 방법 (COUNT)

**배경**: 전체 학생 수와, 키가 기록된 학생 수는 다릅니다. 한 번에 확인합니다.

**요구사항**: `student` 에서 **전체 학생 수** · **키가 기록된 학생 수** · **학생이 속한 학과가 몇 가지인지**를 한 줄로 조회해 `q19` 에 담으세요. 별칭은 차례로 **`total`** · **`with_height`** · **`dept_kinds`** 입니다.

**예시**: `total` 은 32, `with_height` 는 29, `dept_kinds` 는 4 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 행을 세는 것, 값이 있는 행만 세는 것, 서로 다른 값의 가짓수를 세는 것은 모두 다르다
- 셋 다 같은 함수의 괄호 안을 바꿔 표현한다

세부구현:
1. 행 전체를 세는 모양을 첫 열에 적고 별칭을 붙인다
2. 특정 열을 세는 모양을 둘째 열에 적는다
3. 셋째 열은 그 열의 중복을 없앤 뒤 세도록 괄호 안에 키워드를 하나 더 넣는다
```

</details>

In [ ]:
# 세는 방법 세 가지 — count(*) 는 행, count(열) 은 값이 있는 행, count(DISTINCT 열) 은 값의 가짓수
q19 = run_query("""
SELECT count(*)                 AS total,
       count(height)            AS with_height,
       count(DISTINCT dept_id)  AS dept_kinds
FROM student
""")
display(q19)

<details><summary>해설</summary>

- `count(*)` 는 **행의 개수**를 셉니다. 빈 칸이 있어도 셉니다.
- `count(열)` 은 그 열이 **비어 있지 않은 행만** 셉니다. 그래서 둘의 차이가 곧 결측 개수입니다.
- `count(DISTINCT 열)` 은 **서로 다른 값이 몇 가지인지** 셉니다 — `SELECT DISTINCT` 로 목록을 뽑아 세는 것과 결과가 같지만 한 줄로 끝납니다.

</details>

In [ ]:
# [자가채점]
assert list(q19.columns) == ["total", "with_height", "dept_kinds"], \
    f"별칭은 total·with_height·dept_kinds 입니다: {list(q19.columns)}"
assert int(q19["total"][0]) == 32, "total 을 확인하세요"
assert int(q19["with_height"][0]) == 29, "with_height 를 확인하세요"
assert int(q19["dept_kinds"][0]) == 4, \
    "dept_kinds 를 확인하세요 — 중복을 없앤 가짓수여야 합니다"
print("✅ 통과!")

## 20. 합계와 평균 (SUM · AVG)

**배경**: 개설 과목 전체의 학점 규모를 봅니다.

**요구사항**: `subject` 에서 **학점의 합계**와 **평균**을 한 줄로 조회해 `q20` 에 담으세요. 별칭은 각각 **`credit_sum`** 과 **`credit_avg`** 입니다.

**예시**: `credit_sum` 은 27, `credit_avg` 는 2.7 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 합계와 평균을 구하는 집계 함수를 각각 쓴다

세부구현:
1. 두 집계 함수를 콤마로 나란히 적는다
2. 각각에 별칭을 붙인다
```

</details>

In [ ]:
# 집계 함수는 여러 행을 한 값으로 요약한다 — 결과는 언제나 한 줄. avg 는 NULL 을 빼고 계산한다
q20 = run_query("""
SELECT sum(credit) AS credit_sum,
       avg(credit) AS credit_avg
FROM subject
""")
display(q20)

<details><summary>해설</summary>

- 집계 함수는 여러 행을 **한 값**으로 요약합니다. 그래서 결과는 언제나 한 줄입니다.
- `avg` 는 `NULL` 인 값을 계산에서 빼고 평균을 냅니다 — 0 으로 치지 않습니다.

</details>

In [ ]:
# [자가채점]
assert list(q20.columns) == ["credit_sum", "credit_avg"], \
    f"별칭은 credit_sum·credit_avg 입니다: {list(q20.columns)}"
assert int(q20["credit_sum"][0]) == 27, "합계를 확인하세요"
assert abs(float(q20["credit_avg"][0]) - 2.7) < 0.01, "평균을 확인하세요"
print("✅ 통과!")

## 21. 가장 큰 값과 작은 값 (MAX · MIN)

**배경**: 이번엔 학생 키의 최고·최저를 봅니다.

**요구사항**: `student` 에서 **키의 최댓값**과 **최솟값**을 한 줄로 조회해 `q21` 에 담으세요. 별칭은 각각 **`max_height`** 와 **`min_height`** 입니다.

**예시**: `max_height` 는 188.0, `min_height` 는 153.0 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 최댓값·최솟값을 구하는 집계 함수를 각각 쓴다

세부구현:
1. 두 집계 함수를 콤마로 나란히 적고 별칭을 붙인다
```

</details>

In [ ]:
# max·min 도 NULL 을 무시한다 — 키가 비어 있는 학생이 있어도 결과가 흔들리지 않는다
q21 = run_query("""
SELECT max(height) AS max_height,
       min(height) AS min_height
FROM student
""")
display(q21)

<details><summary>해설</summary>

- `max` · `min` 도 `NULL` 을 무시합니다. 그래서 키가 비어 있는 학생이 있어도 결과가 흔들리지 않습니다.
- 글자 열에도 쓸 수 있습니다 — 사전 순으로 가장 앞·뒤 값을 돌려줍니다.

</details>

In [ ]:
# [자가채점]
assert list(q21.columns) == ["max_height", "min_height"], \
    f"별칭은 max_height·min_height 입니다: {list(q21.columns)}"
assert abs(float(q21["max_height"][0]) - 188.0) < 0.01, "최댓값을 확인하세요"
assert abs(float(q21["min_height"][0]) - 153.0) < 0.01, "최솟값을 확인하세요"
print("✅ 통과!")

## 22. 그룹별로 세기 (GROUP BY)

**배경**: 학과별 학생 수를 냅니다.

**요구사항**: `student` 를 **학과별로 묶어** 학생 수를 세고, **학생이 많은 학과부터** 정렬해 `q22` 에 담으세요. 열은 `dept_id` 와 학생 수(별칭 **`student_count`**) 입니다.

**예시**: 결과는 4행이고, 첫 행은 `CS` 학과 10명입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 값을 가진 행끼리 묶어 묶음마다 세는 문법을 쓴다

세부구현:
1. SELECT 에 기준 열과 세는 집계 함수를 적는다
2. GROUP BY 로 묶을 기준 열을 지정한다
3. ORDER BY 로 개수 내림차순 정렬한다
```

</details>

In [ ]:
# 묶음 하나가 결과 한 행이 된다. SELECT 에는 묶은 기준 열과 집계 함수만 올 수 있다
q22 = run_query("""
SELECT dept_id,
       count(*) AS student_count
FROM student
GROUP BY dept_id
ORDER BY student_count DESC
""")
display(q22)

<details><summary>해설</summary>

- `GROUP BY` 로 묶으면 **묶음 하나가 결과 한 행**이 됩니다.
- `SELECT` 에 적을 수 있는 것은 **묶은 기준 열**과 **집계 함수** 뿐입니다. `stu_name` 처럼 묶음 안에서 값이 여럿인 열을 적으면 의미가 모호해집니다.
- 별칭 `student_count` 는 `ORDER BY` 에서 쓸 수 있습니다.

</details>

In [ ]:
# [자가채점]
assert len(q22) == 4, f"4행이어야 합니다 (지금 {len(q22)}행)"
assert list(q22.columns) == ["dept_id", "student_count"], \
    f"열은 dept_id·student_count 입니다: {list(q22.columns)}"
assert q22["dept_id"][0] == "CS", "학생이 가장 많은 학과가 첫 행이어야 합니다"
assert int(q22["student_count"][0]) == 10, "CS 학과 학생 수를 확인하세요"
counts = q22["student_count"].tolist()
assert counts == sorted(counts, reverse=True), \
    f"학생이 많은 학과부터 정렬하세요: {counts}"
print("✅ 통과!")

## 23. 과목별 수강 인원 (GROUP BY — 다른 표)

**배경**: 강의실 배정을 위해 과목마다 몇 명이 듣는지 셉니다.

**요구사항**: `enrol` 표를 **과목별로 묶어** 수강 인원을 세고, **인원이 많은 과목부터**, 같으면 **과목 번호 오름차순**으로 정렬해 `q23` 에 담으세요. 열은 `sub_no` 와 인원(별칭 **`enrol_count`**) 입니다.

**예시**: 첫 행은 과목 `104` 이고 인원은 16명입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 앞 문제와 같은 구조를, 중간표에 적용한다

세부구현:
1. 기준 열과 세는 집계 함수를 적는다
2. GROUP BY 로 과목 번호를 지정한다
3. ORDER BY 에 기준 두 개를 적는다 (인원 내림차순, 과목 번호 오름차순)
```

</details>

In [ ]:
# 학생과 과목을 잇는 중간표를 과목별로 세면 그것이 곧 수강 인원이 된다
q23 = run_query("""
SELECT sub_no,
       count(*) AS enrol_count
FROM enrol
GROUP BY sub_no
ORDER BY enrol_count DESC, sub_no
""")
display(q23)

<details><summary>해설</summary>

- `enrol` 은 학생과 과목을 잇는 중간표라, 여기서 세면 곧 **수강 인원**이 됩니다.
- 정렬 기준이 둘일 때 방향은 각각 따로 정합니다 — 앞은 `DESC`, 뒤는 기본(오름차순).

</details>

In [ ]:
# [자가채점]
assert len(q23) == 10, f"10행이어야 합니다 (지금 {len(q23)}행)"
assert list(q23.columns) == ["sub_no", "enrol_count"], \
    f"열은 sub_no·enrol_count 입니다: {list(q23.columns)}"
assert q23["sub_no"].tolist()[:5] == ['104', '103', '105', '108', '109'], \
    f"인원 내림차순, 같으면 과목 번호 오름차순이어야 합니다: {q23['sub_no'].tolist()[:5]}"
assert int(q23["enrol_count"][0]) == 16, "104 과목 인원을 확인하세요"
assert int(q23["enrol_count"].sum()) == 114, "인원의 합이 전체 수강 건수와 같아야 합니다"
print("✅ 통과!")

## 24. 값 수정 (UPDATE)

**배경**: 데이터분석회의 인기가 높아 정원을 늘리기로 했습니다.

**요구사항**: `club` 표에서 **`club_name` 이 `'데이터분석회'`** 인 동아리의 `max_member` 를 **30** 으로 바꾸세요. 바꾸기 전에 **같은 조건으로 `SELECT` 를 먼저** 실행해 대상을 확인하는 습관을 지키세요.

**예시**: 수정 후 그 동아리의 `max_member` 는 30 이고, 다른 동아리의 정원은 그대로입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 바꿀 대상을 조건으로 콕 집어야 한다. 조건을 빼면 표 전체가 바뀐다

세부구현:
1. 같은 조건으로 SELECT 를 먼저 실행해 바뀔 행을 눈으로 확인한다
2. UPDATE 표 SET 열 = 값 WHERE 조건 순서로 적는다
3. 다시 조회해 바뀐 값을 확인한다
```

</details>

In [ ]:
# 1) 바꾸기 전에 같은 조건으로 조회 — 무엇이 바뀔지 눈으로 확인한다
display(run_query("SELECT * FROM club WHERE club_name = '데이터분석회'"))

# 2) 그 다음에 바꾼다. WHERE 를 빠뜨리면 모든 동아리의 정원이 바뀐다
run_sql("UPDATE club SET max_member = 30 WHERE club_name = '데이터분석회'")

# 3) 다시 조회해 의도한 행만 바뀌었는지 확인한다
display(run_query("SELECT club_name, max_member FROM club ORDER BY club_id"))

<details><summary>해설</summary>

- `WHERE` 를 빠뜨리면 **모든 동아리의 정원**이 30 이 됩니다. 되돌리기도 어렵습니다.
- 실수를 되돌릴 수 있게 `run_sql("BEGIN")` 으로 묶고 결과를 본 뒤 `COMMIT` 또는 `ROLLBACK` 하는 방법도 있습니다.

</details>

In [ ]:
# [자가채점]
rows = run_query("SELECT club_name, max_member FROM club ORDER BY club_id")
target = rows[rows["club_name"] == "데이터분석회"]
assert len(target) == 1, "데이터분석회 가 한 행 있어야 합니다"
assert int(target["max_member"].iloc[0]) == 30, "데이터분석회의 정원은 30 이어야 합니다"
others = rows[rows["club_name"] != "데이터분석회"]
assert sorted(others["max_member"].tolist()) == [15, 20], \
    "다른 동아리의 정원까지 바뀌었습니다 — WHERE 조건을 확인하세요"

# 25번 채점이 '그 뒤로 실제로 몇 행이 바뀌었는지' 재려고 지금 시점을 기록해 둡니다.
# (리셋 셀은 연결을 새로 만들어 이 숫자가 0부터 다시 세어지므로, 연결도 함께 기억합니다.)
conn_before_25 = get_conn()
changes_before_25 = conn_before_25.total_changes
print("✅ 통과!")

## 25. 되돌리고, 다시 지워 확정하기 (ROLLBACK · COMMIT)

**배경**: 로봇제작반이 해체되어 명단에서 지웁니다. 그런데 `DELETE` 는 조건을 빠뜨리면 **표를 통째로 비웁니다.** 그 사고를 **일부러 한 번 내 보고 되돌린 뒤**, 제대로 지워 확정합니다.

**요구사항**: 아래 다섯 단계를 순서대로 실행하세요.

1. `run_sql("BEGIN")` 으로 트랜잭션을 엽니다.
2. **조건 없이** `DELETE FROM club` 을 실행합니다(사고 재현). 곧바로 남은 행 수를 세어 **`during`** 에 담으세요.
3. `run_sql("ROLLBACK")` 으로 되돌리고, 남은 행 수를 세어 **`after_rollback`** 에 담으세요.
4. 다시 `BEGIN` 을 열고, 이번엔 **`club_name` 이 `'로봇제작반'`** 인 행만 지웁니다.
5. 남은 행을 확인하고 `run_sql("COMMIT")` 으로 확정합니다.

**예시**: `during` 은 **0**(전부 지워진 상태), `after_rollback` 은 **3**(되돌아옴), 확정 후 `club` 은 **2행**이 남습니다.

> 행 수를 세는 것은 `run_query("SELECT count(*) AS n FROM club")["n"][0]` 로 얻습니다.

> **왜 이렇게 하나**: `ROLLBACK` 은 말로만 들으면 와닿지 않습니다. **전부 지워진 0 을 직접 보고**, 되돌린 뒤 3 으로 돌아오는 것을 확인해야 "묶어 두면 되돌릴 수 있다"가 몸에 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 트랜잭션을 열면 그 뒤의 변경은 확정 전까지 임시다. 되돌리는 명령과 확정하는 명령이 따로 있다
- 되돌리기 전에 '지금 상태'를 세어 두어야 되돌아온 것을 증명할 수 있다
- 한 번 닫은 트랜잭션은 다시 열어야 한다

세부구현:
1. 트랜잭션을 연다
2. 조건 없는 DELETE 를 실행하고 남은 행 수를 during 에 담는다
3. 되돌리는 명령을 실행하고 다시 센 값을 after_rollback 에 담는다
4. 트랜잭션을 다시 열고 조건을 붙인 DELETE 를 실행한다
5. 남은 행을 조회한 뒤 확정하는 명령을 실행한다
```

</details>

In [ ]:
# 1) 트랜잭션을 연다 — 여기부터의 변경은 확정 전까지 임시다
run_sql("BEGIN")

# 2) 사고 재현: 조건을 빠뜨린 DELETE 는 표를 통째로 비운다
run_sql("DELETE FROM club")
during = run_query("SELECT count(*) AS n FROM club")["n"][0]
print("조건 없이 지운 직후 남은 행:", during)

# 3) 되돌린다 — 확정하지 않았으므로 없던 일이 된다
run_sql("ROLLBACK")
after_rollback = run_query("SELECT count(*) AS n FROM club")["n"][0]
print("되돌린 뒤 남은 행:", after_rollback)

# 4) 이번엔 조건을 붙여 제대로 지운다 (트랜잭션은 다시 열어야 한다)
run_sql("BEGIN")
run_sql("DELETE FROM club WHERE club_name = '로봇제작반'")
display(run_query("SELECT * FROM club ORDER BY club_id"))

# 5) 의도대로면 확정한다 — 묶었으면 반드시 닫는다
run_sql("COMMIT")

<details><summary>해설</summary>

- `BEGIN` 뒤의 변경은 **아직 임시**입니다. `COMMIT` 으로 확정하거나 `ROLLBACK` 으로 되돌립니다. 이렇게 묶은 한 덩어리를 **트랜잭션**이라고 합니다.
- 2단계에서 본 `0` 이 이 문제의 요점입니다. **조건을 빠뜨린 `DELETE` 는 경고 없이 전부 지웁니다.** 묶어 두지 않았다면 그대로 끝이었습니다.
- `ROLLBACK` 은 **확정 전**에만 통합니다. `COMMIT` 뒤에는 되돌릴 수 없습니다 — 그래서 확인은 항상 `COMMIT` 앞에서 합니다.
- `COMMIT`·`ROLLBACK` 은 트랜잭션을 **닫습니다.** 그래서 4단계에서 `BEGIN` 을 다시 열었습니다. 닫지 않고 다음 문제로 가면 잠금이 남아 나중에 헷갈립니다.
- 표 자체를 없애려면 `DROP TABLE club` 입니다 — 지우는 범위가 다릅니다.

</details>

In [ ]:
# [자가채점]
assert int(during) == 0, \
    f"during 은 조건 없이 지운 직후의 행 수(0)여야 합니다 — 지금 {during}"
assert int(after_rollback) == 3, \
    f"after_rollback 은 되돌린 뒤의 행 수(3)여야 합니다 — ROLLBACK 이 빠졌는지 확인하세요 (지금 {after_rollback})"

rows = run_query("SELECT club_name FROM club")
assert len(rows) == 2, f"확정 후 2행이 남아야 합니다 (지금 {len(rows)}행)"
assert "로봇제작반" not in rows["club_name"].tolist(), "로봇제작반 이 아직 남아 있습니다"
assert set(rows["club_name"]) == {"알고리즘연구회", "데이터분석회"}, \
    "다른 동아리까지 지워졌습니다 — WHERE 조건을 확인하세요"

# during·after_rollback 은 값만 적어 넣을 수도 있습니다. 그래서 '실제로 몇 행을 건드렸는지'를
# 함께 봅니다 — ROLLBACK 은 되돌리지만 '바꾼 횟수'까지 되돌리지는 않습니다.
assert "changes_before_25" in dir(), \
    "24번 자가채점을 먼저 실행하세요 — 이 채점이 그 시점을 기준으로 삼습니다"

changed = get_conn().total_changes - changes_before_25
assert conn_before_25 is get_conn() and changed >= 0, \
    ("리셋 셀을 실행한 뒤라 기준이 맞지 않습니다 — 24번 답안과 자가채점부터 다시 "
     "실행하세요. 리셋하면 club 표도 사라지므로 1~4번을 먼저 다시 푼 뒤 "
     "24번, 25번 순서로 오세요")
assert changed >= 4, \
    ("전체 삭제 -> ROLLBACK -> 조건부 삭제 흐름을 실제로 거치지 않았습니다 "
     "(값만 적어 넣지 말고 단계를 실행하세요). 24번 채점을 다시 돌렸다면 "
     "25번 답안 셀부터 다시 실행하세요)")

# 트랜잭션이 열린 채 남아 있지 않은지 확인합니다 (COMMIT 을 했다면 새로 BEGIN 이 됩니다).
try:
    run_sql("BEGIN")
    run_sql("ROLLBACK")
except sqlite3.OperationalError:
    raise AssertionError("트랜잭션이 아직 열려 있습니다 — 마지막에 COMMIT 을 실행하세요")

print("✅ 통과!")

## 26. ERD 읽기 (서술)

**배경**: 아래는 이 실습 데이터의 구조를 그린 ERD 입니다.

![학사 ERD](../../day17_데이터베이스_SQL/images/erd_학사.png)

**요구사항**: 아래 세 가지를 자신의 말로 적으세요. (코드는 필요 없습니다.)

1. `department` 와 `student` 는 몇 대 몇 관계이고, 외래키가 왜 `student` 쪽에 있는지
2. `student` 와 `subject` 는 원래 몇 대 몇 관계이며, `enrol` 이 왜 필요한지
3. `enr_grade`(성적)를 `student` 에 두면 무엇이 곤란하고, `subject` 에 두면 무엇이 곤란한지 — 각각 한 문장씩

> 정답은 정답 노트북의 **모범 서술**과 비교해 확인하세요.

**모범 서술**

**1. department : student = 1 : N**

한 학과에 학생이 여럿 속하고, 한 학생은 한 학과에만 속합니다. 이때 외래키는 **언제나 N 쪽**에 둡니다 — 학생 한 명은 가리킬 학과가 하나뿐이라 `student.dept_id` 열 하나면 충분하기 때문입니다. 반대로 `department` 쪽에 두려면 학과마다 학생 번호를 여러 개 담아야 하는데, 한 칸에 값 여러 개를 넣는 것은 표가 지켜야 할 규칙을 깨는 일입니다.

**2. student : subject = M : N — 그래서 `enrol` 이 필요합니다**

한 학생이 과목을 여럿 듣고, 한 과목도 학생을 여럿 받습니다. 관계형 표는 이 관계를 선 하나로 그릴 수 없습니다. 어느 쪽에 외래키를 둬도 값이 여러 개가 되기 때문입니다. 그래서 **중간표** `enrol` 을 하나 두고 `student` 1:N `enrol` N:1 `subject` 로 **1:N 두 개로 나눕니다.** `enrol` 의 기본키는 `(sub_no, stu_no)` 두 열을 묶은 것이라, 같은 학생이 같은 과목을 두 번 신청하는 일도 자동으로 막힙니다.

**3. `enr_grade` 는 관계에만 있는 정보입니다**

성적은 '학생의 성질'도 '과목의 성질'도 아니고, **그 학생이 그 과목을 들었을 때** 비로소 생기는 값입니다. `student` 에 두면 과목마다 다른 성적을 담을 자리가 없고, `subject` 에 두면 학생마다 다른 성적을 담을 자리가 없습니다. 이렇게 **두 개체가 만나는 자리에서만 생기는 정보**는 중간표에 둡니다. 학기·신청일 같은 값도 마찬가지입니다.

## 다 풀었다면

- 자가채점이 모두 `✅ 통과!` 인지 확인하세요.
- 26번 서술은 정답 노트북의 모범 서술과 비교해 빠진 관점이 없는지 보세요.
- 이어서 **과제 LV2** 로 넘어갑니다 — `JOIN` · `HAVING` · 서브쿼리로 표 여럿을 한 문장에 담습니다.